In [1]:
!pip install -q transformers>=4.40.0 peft>=0.10.0 datasets accelerate bitsandbytes trl scipy sentencepiece protobuf

import torch
print(f"PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

PyTorch: 2.10.0+cu128, CUDA available: True


In [2]:
import os, json, zipfile
from google.colab import files

from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/processed.zip /content/

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile("/content/processed.zip", 'r') as z:
    z.extractall(DATA_DIR)

print("Extracted files:")
for f in sorted(os.listdir(DATA_DIR)):
    print(f"  {f}")

Mounted at /content/drive
Extracted files:
  aqua_rat_test.jsonl
  aqua_rat_train.jsonl
  aqua_rat_validation.jsonl
  gsm8k_test.jsonl
  gsm8k_train.jsonl
  gsm8k_validation.jsonl
  merged_test.jsonl
  merged_train.jsonl
  merged_validation.jsonl
  svamp_test.jsonl
  svamp_train.jsonl
  svamp_validation.jsonl


In [21]:
def load_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

# Load all splits
train_data = load_jsonl(os.path.join(DATA_DIR, "merged_train.jsonl"))
val_data = load_jsonl(os.path.join(DATA_DIR, "merged_validation.jsonl"))

# Load individual test sets for per-dataset evaluation
test_gsm8k = load_jsonl(os.path.join(DATA_DIR, "gsm8k_test.jsonl"))
test_svamp = load_jsonl(os.path.join(DATA_DIR, "svamp_test.jsonl"))
test_aqua = load_jsonl(os.path.join(DATA_DIR, "aqua_rat_test.jsonl"))

print(f"Train: {len(train_data)} | Val: {len(val_data)}")
print(f"Test - GSM8K: {len(test_gsm8k)} | SVAMP: {len(test_svamp)} | AQuA-RAT: {len(test_aqua)}")

Train: 104719 | Val: 1071
Test - GSM8K: 300 | SVAMP: 300 | AQuA-RAT: 254


In [7]:
def format_prompt(tokenizer,sample):
    """Build the instruction prompt using the SAME chat template as baseline."""
    question = sample["question"].strip()
    messages = [
        {"role": "system",
         "content": "You are a helpful and precise assistant. Please solve the following problem step by step. End your response by clearly stating the final answer."},
        {"role": "user", "content": question}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def format_completion(sample):
    """Build the expected completion (target output)."""
    rationale = sample.get("rationale", "") or ""
    answer = sample["answer"].strip()
    rationale = rationale.strip().rstrip('"').strip()
    if rationale:
        return f"{rationale}\nThe answer is: {answer}<|im_end|>"
    else:
        return f"The answer is: {answer}<|im_end|>"



In [22]:
import random
random.seed(42)

TRAIN_BUDGET = 8000

train_by_ds = {}
for s in train_data:
    ds = s["dataset"]
    train_by_ds.setdefault(ds, []).append(s)

per_ds = TRAIN_BUDGET // len(train_by_ds)
train_subset = []
for ds, items in train_by_ds.items():
    n = min(per_ds, len(items))
    train_subset.extend(random.sample(items, n))

random.shuffle(train_subset)
val_subset = random.sample(val_data, min(500, len(val_data)))

print(f"Subsampled training data:")
for ds, items in train_by_ds.items():
    sampled = min(per_ds, len(items))
    print(f"  {ds}: {sampled}")
print(f"  Total: {len(train_subset)} train, {len(val_subset)} val")

Subsampled training data:
  aqua_rat: 2666
  gsm8k: 2666
  svamp: 630
  Total: 5962 train, 500 val


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ============================================================
#  Qwen2.5 Config
# ============================================================
cfg = {
    "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
    "target_modules": ["q_proj", "v_proj"],
}

ACTIVE_MODEL = "qwen2.5"
print(f"Active model: {cfg['model_name']}")


Active model: Qwen/Qwen2.5-1.5B-Instruct


In [12]:
# ============================================================
#  Load Tokenizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    cfg["model_name"],
    trust_remote_code=True,
)

# Ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size: 151643
Pad token: <|endoftext|> (id=151643)


In [13]:
# ============================================================
#  Load Base Model
# ============================================================
base_model = AutoModelForCausalLM.from_pretrained(
    cfg["model_name"],
    trust_remote_code=True,
    torch_dtype=torch.float16,
).to("cuda")

# Print model size info
total_params = sum(p.numel() for p in base_model.parameters())
print(f"Base model loaded: {total_params / 1e6:.1f}M parameters")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded: 1543.7M parameters
GPU memory: 2.88 GB


In [14]:
# ============================================================
#  Apply LoRA (conservative for Qwen - already strong baseline)
# ============================================================
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=cfg["target_modules"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [23]:
from datasets import Dataset

MAX_LENGTH = 512

def tokenize_sample(sample):
    prompt = format_prompt(tokenizer,sample)
    completion = format_completion(sample)
    full_text = prompt + completion

    # Tokenize full text
    full_enc = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    # Tokenize prompt only to find where completion starts
    prompt_enc = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    prompt_len = len(prompt_enc["input_ids"])

    # Build labels: -100 for prompt tokens, actual ids for completion tokens
    labels = [-100] * prompt_len + full_enc["input_ids"][prompt_len:]

    return {
        "input_ids": full_enc["input_ids"],
        "attention_mask": full_enc["attention_mask"],
        "labels": labels,
    }


# Process datasets
print("Tokenizing training data...")
train_tokenized = [tokenize_sample(s) for s in train_subset]
print("Tokenizing validation data...")
val_tokenized = [tokenize_sample(s) for s in val_subset]

train_dataset = Dataset.from_list(train_tokenized)
val_dataset = Dataset.from_list(val_tokenized)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

# Check token length distribution
lengths = [len(x["input_ids"]) for x in train_tokenized]
print(f"Token lengths - min: {min(lengths)}, max: {max(lengths)}, mean: {sum(lengths)/len(lengths):.0f}")

Tokenizing training data...
Tokenizing validation data...
Train dataset: 5962 samples
Val dataset: 500 samples
Token lengths - min: 80, max: 512, mean: 205


## Training

In [24]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import time

OUTPUT_DIR = f"/content/lora_{ACTIVE_MODEL}"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print(f"Starting training for {ACTIVE_MODEL}...")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training for qwen2.5...
  Epochs: 2
  Effective batch size: 16
  Learning rate: 5e-05
  LoRA rank: 16, alpha: 32


In [25]:
# ============================================================
#  Train & Record Metrics
# ============================================================
torch.cuda.reset_peak_memory_stats()
start_time = time.time()

train_result = trainer.train()

train_time = time.time() - start_time
peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3

print(f"\n{'='*50}")
print(f"Training completed for: {ACTIVE_MODEL}")
print(f"  Total time: {train_time/60:.1f} minutes")
print(f"  Peak GPU memory: {peak_memory_gb:.2f} GB")
print(f"  Final train loss: {train_result.training_loss:.4f}")
print(f"{'='*50}")

Step,Training Loss,Validation Loss
200,0.619651,0.511769
400,0.611265,0.496726
600,0.586270,0.493924



Training completed for: qwen2.5
  Total time: 8.3 minutes
  Peak GPU memory: 12.16 GB
  Final train loss: 0.6359


In [26]:
# Save the LoRA adapter
SAVE_PATH = f"/content/lora_adapter_{ACTIVE_MODEL}"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"LoRA adapter saved to: {SAVE_PATH}")

# Also save training metrics
metrics = {
    "model": cfg["model_name"],
    "active_model": ACTIVE_MODEL,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "train_samples": len(train_subset),
    "epochs": int(training_args.num_train_epochs),
    "effective_batch_size": training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
    "learning_rate": training_args.learning_rate,
    "train_loss": round(train_result.training_loss, 4),
    "train_time_minutes": round(train_time / 60, 1),
    "peak_gpu_memory_gb": round(peak_memory_gb, 2),
}

with open(f"/content/train_metrics_{ACTIVE_MODEL}.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

LoRA adapter saved to: /content/lora_adapter_qwen2.5
{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "active_model": "qwen2.5",
  "lora_r": 16,
  "lora_alpha": 32,
  "train_samples": 5962,
  "epochs": 2,
  "effective_batch_size": 16,
  "learning_rate": 5e-05,
  "train_loss": 0.6359,
  "train_time_minutes": 8.3,
  "peak_gpu_memory_gb": 12.16
}


## Evaluation on Test Sets

In [27]:
import re

def extract_answer_from_output(response_text: str, ground_truth: str):
    """Use the SAME extraction logic as baseline for fair comparison."""
    response_text = response_text.strip()
    gt_str = str(ground_truth).strip().upper()

    if gt_str in ['A', 'B', 'C', 'D', 'E']:
        patterns = [
            r'(?i)(?:answer|option)(?:\s+is)?(?:\s*:)?\s*([A-E])\b',
            r'(?i)the correct answer is\s*([A-E])\b',
            r'(?i)therefore,.*?\b([A-E])\b',
            r'(?:^|\s)([A-E])(?:\s|$|\.|\,|:)'
        ]
        for pattern in patterns:
            matches = re.findall(pattern, response_text)
            if matches:
                return matches[-1].upper(), gt_str
        return "NONE", gt_str
    else:
        text = response_text.replace(',', '').replace('**', '').replace('$', '')
        patterns = [
            r'(?i)answer is\s*[:=]*\s*(-?\d+\.?\d*)',
            r'(?i)answer:\s*(-?\d+\.?\d*)',
        ]
        try:
            label_num = float(str(ground_truth).replace(',', '').replace('$', ''))
        except ValueError:
            label_num = float('inf')

        for pattern in patterns:
            matches = re.findall(pattern, text)
            if matches:
                try:
                    return float(matches[-1]), label_num
                except ValueError:
                    continue

        pred = re.findall(r'-?\d+\.?\d*', text)
        if not pred:
            return float('inf'), label_num
        try:
            return float(pred[-1]), label_num
        except ValueError:
            return float('inf'), label_num


def evaluate_model(model, tokenizer, test_data, dataset_name, max_new_tokens=512):
    """Evaluate model using the SAME generation config as baseline."""
    model.eval()
    correct = 0
    total = len(test_data)
    results = []
    miss = 1e-4

    for i, sample in enumerate(test_data):
        prompt = format_prompt(tokenizer,sample)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )

        generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        pred_val, label_val = extract_answer_from_output(generated, sample["answer"])

        if isinstance(pred_val, str):
            is_correct = (pred_val == label_val)
        else:
            is_correct = (abs(label_val - pred_val) <= miss)

        if is_correct:
            correct += 1

        results.append({
            "id": sample["id"],
            "question": sample["question"],
            "gold": sample["answer"],
            "pred": str(pred_val),
            "generated_text": generated[:300],
            "correct": is_correct,
        })

        if (i + 1) % 50 == 0:
            print(f"  [{dataset_name}] {i+1}/{total} done, running acc: {correct/(i+1)*100:.1f}%")

    accuracy = correct / total * 100
    print(f"  [{dataset_name}] Final Accuracy: {accuracy:.2f}% ({correct}/{total})")
    return accuracy, results


In [28]:
# ============================================================
#  Run Evaluation on All 3 Test Sets
# ============================================================
print(f"Evaluating LoRA-finetuned {ACTIVE_MODEL}...\n")

acc_gsm8k, results_gsm8k = evaluate_model(model, tokenizer, test_gsm8k, "GSM8K")
acc_svamp, results_svamp = evaluate_model(model, tokenizer, test_svamp, "SVAMP")
acc_aqua, results_aqua = evaluate_model(model, tokenizer, test_aqua, "AQuA-RAT")

print(f"\n{'='*50}")
print(f"  PEFT Results for: {ACTIVE_MODEL}")
print(f"  GSM8K:    {acc_gsm8k:.2f}%")
print(f"  SVAMP:    {acc_svamp:.2f}%")
print(f"  AQuA-RAT: {acc_aqua:.2f}%")
print(f"{'='*50}")

Evaluating LoRA-finetuned qwen2.5...

  [GSM8K] 50/300 done, running acc: 52.0%
  [GSM8K] 100/300 done, running acc: 49.0%
  [GSM8K] 150/300 done, running acc: 50.7%
  [GSM8K] 200/300 done, running acc: 51.0%
  [GSM8K] 250/300 done, running acc: 50.8%
  [GSM8K] 300/300 done, running acc: 50.3%
  [GSM8K] Final Accuracy: 50.33% (151/300)
  [SVAMP] 50/300 done, running acc: 50.0%
  [SVAMP] 100/300 done, running acc: 50.0%
  [SVAMP] 150/300 done, running acc: 52.0%
  [SVAMP] 200/300 done, running acc: 51.5%
  [SVAMP] 250/300 done, running acc: 55.2%
  [SVAMP] 300/300 done, running acc: 54.0%
  [SVAMP] Final Accuracy: 54.00% (162/300)
  [AQuA-RAT] 50/254 done, running acc: 50.0%
  [AQuA-RAT] 100/254 done, running acc: 50.0%
  [AQuA-RAT] 150/254 done, running acc: 49.3%
  [AQuA-RAT] 200/254 done, running acc: 45.5%
  [AQuA-RAT] 250/254 done, running acc: 44.8%
  [AQuA-RAT] Final Accuracy: 45.67% (116/254)

  PEFT Results for: qwen2.5
  GSM8K:    50.33%
  SVAMP:    54.00%
  AQuA-RAT: 45.67%


In [29]:
# Save evaluation results
eval_results = {
    "model": cfg["model_name"],
    "method": "LoRA",
    "accuracy": {
        "gsm8k": round(acc_gsm8k, 2),
        "svamp": round(acc_svamp, 2),
        "aqua_rat": round(acc_aqua, 2),
    },
    "lora_config": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "target_modules": cfg["target_modules"],
    },
}

with open(f"/content/eval_results_{ACTIVE_MODEL}.json", "w") as f:
    json.dump(eval_results, f, indent=2)

with open(f"/content/predictions_{ACTIVE_MODEL}_gsm8k.json", "w") as f:
    json.dump(results_gsm8k, f, indent=2)
with open(f"/content/predictions_{ACTIVE_MODEL}_svamp.json", "w") as f:
    json.dump(results_svamp, f, indent=2)
with open(f"/content/predictions_{ACTIVE_MODEL}_aqua.json", "w") as f:
    json.dump(results_aqua, f, indent=2)

print("All results saved!")
print(json.dumps(eval_results, indent=2))

All results saved!
{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "method": "LoRA",
  "accuracy": {
    "gsm8k": 50.33,
    "svamp": 54.0,
    "aqua_rat": 45.67
  },
  "lora_config": {
    "r": 16,
    "alpha": 32,
    "target_modules": [
      "q_proj",
      "v_proj"
    ]
  }
}
